# Reto 1

## Obtención del Dataset
Para poder hacer uso del dataset MASATI V2 primero debe obtenerse a través de un formulario de google donde al responder lo enviaran al correo.

## Análisis del dataset
Con el dataset obtenido ya se puede empezar el análisis del siguiente, el cual cuenta con la siguiente estructura:
```bash
.
├── coast (imagenes de costas sin barcos)
├── coast_ship (imagenes de costas con barcos)
├── coast_ship_labels (archivos que contienen las coordenadas de cada barco en de las costas)
├── detail (imagenes de barcos aumentada)
├── land (imágenes de tierra)
├── multi (imagenes con múltiples barcos)
├── multi_labels (archivos de coordenadas de los múltiples barcos)
├── ship (imágenes barcos solos)
├── ship_labels (archivos de coordenada de los barcos)
└── water (imágenes de cuerpos de agua vacios)
```

Cada carpeta contiene imágenes de barcos o de cuerpos de agua / costas donde podria haber un barco.

Como se puede ver hay carpetas que repiten el nombre con "_labels" al final, las carpetas que terminan en "_labels" tienen la misma cantidad de archivos y con los mismos nombre (sin tomar en cuenta la extensión) que la otra carpeta sin "_labels" en el nombre. Las carpetas que terminan en "_labelsl" tienen archivos xml que contienen las coordenadas de cada recuadro que encierra un barco de las imágenes que los contengan.

## Implementación del reto
Con esta información sobre la estructura de carpetas y archivos del dataset se puede crear un código en python para buscar en las carpetas imágenes aleatorias de barcos y posteriormente con cv2 dibujar un recuadro a su alrededor con la información de su respectivo archivo xml.


In [3]:
import os
import random
import cv2
import xml.etree.ElementTree as ET
import numpy as np

# Get the dataset root directory path
dataset_base_path = os.path.join(os.path.expanduser('~') ,"Downloads/MASATI-v2")
masati_content = os.listdir(dataset_base_path)

# find folders with labels
with_labels = {}
for folder in masati_content: 
    if folder.endswith("_labels"):
        with_labels[folder.replace("_labels", "")] = folder

# Randomly chose an image and draw a rectangle arround the ship
for data, label in with_labels.items():
    data_path = os.path.join(dataset_base_path, data)
    label_path = os.path.join(dataset_base_path, label)
    image = random.choice(os.listdir(data_path))

    # Scan the xml to get the cords
    tree = ET.parse(os.path.join(label_path, image.replace(".png", ".xml")))
    root = tree.getroot()
    rectangles = []
    for obj in root.findall("object"):
        bbox = obj.find("bndbox")

        xmin = int(bbox.find("xmin").text)
        ymin = int(bbox.find("ymin").text)
        xmax = int(bbox.find("xmax").text)
        ymax = int(bbox.find("ymax").text)

        rectangles.append((xmin, ymin, xmax, ymax))

    # draw every finded rectangle 
    cv2_img = cv2.imread(os.path.join(data_path, image))
    img = cv2_img.copy()
    for (x1,y1,x2,y2) in rectangles:
        cv2.rectangle(img, (x1,y1), (x2, y2), (0,0,255), 1)

    # Show the modify ship image
    cv2.imshow(image, img)
    cv2.waitKey(0)
    cv2.destroyAllWindows()
